# Inference-Time Scaling: From Repeated Sampling to Architecture Search
*Scaling laws for test-time compute, the generation-verification gap, process vs. outcome reward models, sequential revision, and Archon's inference-time architecture search*

# The Three Stages of LLM Development
- **Pre-training** — historically the most time- and compute-intensive stage (can take months, uses huge GPU fleets, trillions of tokens).
- **Fine-tuning** — orders of magnitude less data and compute than pre-training.
- **Inference** — where the trained model is actually used. Historically treated as a single, cheap forward pass per query — but the idea here is spending *more* compute here, without touching model parameters or doing any fine-tuning, to make the model better and more useful.


# Repeated Sampling on Agentic Benchmarks
**Paper:** [arxiv.org/abs/2407.21787](https://arxiv.org/abs/2407.21787)

Beyond math and coding, repeated sampling (recap: generate many candidate responses, then use a verifier to pick the correct one) also works on **agentic benchmarks** like **SWE-bench**, which mimics a software engineer editing code and creating patches.

According to the paper, on SWE-bench Lite, **DeepSeek-V2-Coder-Instruct** went from solving **15.9%** of issues with a single attempt to **56%** with 250 samples — beating the best single-attempt result at the time (43%, from more capable frontier models). Since unit tests can automatically pick out the correct sample, this gives a fully automated way to turn a cheaper open-source model into a system that's actually more capable than models that beat it on a single try.

*(Note: the exact model names and sample counts can sometimes differ between an original paper and a later presentation of it, if a newer internal update was being cited — the numbers above are the ones confirmed in the published paper.)*


# Inference-Time Scaling Laws
Pre-training has well-known scaling laws — test loss decreases predictably as data, compute, and parameters increase. This same paper shows an analogous scaling law exists for **inference-time compute**.

## The Power Law
Coverage (fraction of problems solved by ≥1 sample) follows an exponential power law in the number of parallel samples drawn:

$$\text{Coverage}(k) \approx a \cdot k^{b}$$

where $k$ is the number of samples, and $a, b$ are coefficients fit via curve-fitting to the model's observed scaling behavior.

## Empirical Support
Tested across models spanning **70M to 70B parameters** (Llama 3 8B/70B, Gemma, Pythia, etc.), the fitted power-law curve closely tracks actual observed coverage as sample count increases — including for very small models (e.g., 70M parameters), and consistently across different task domains.

**Practical implication:** given this law, one can predict how many samples (and how much compute) are needed to hit a target coverage level, rather than guessing.


# Why a Power Law? The Long Tail of Hard Problems

## Per-Problem Math
If $p$ is the pass@1 probability (probability a single sample is correct) for an individual problem $i$, then the probability that **at least one** of $k$ samples is correct is:

$$\text{pass@k}_i = 1 - (1-p)^k$$

— derived directly from $(1-p)^k$ being the probability that all $k$ attempts fail.

## From Per-Problem to Population-Level Power Law
This per-problem formula is a simple exponential, not itself a power law. The **population-level power law** (aggregating coverage across an entire dataset of problems) emerges only under a specific condition: the dataset must contain a **long tail of hard problems** — i.e., pass@1 probability distributed with many easy problems (high pass@1) and progressively fewer, harder problems (low pass@1), stretching out into a long tail.

Empirically, this condition holds across the benchmarks studied: a large share of problems are solved at pass@1 (simple problems), while an increasingly small share of harder problems have progressively lower pass@1 — and it's precisely this distribution shape that produces the observed power-law scaling of coverage vs. sample count.


# What This Means for LLM Engineering
Historically, compute investment was heavily front-loaded: hundreds of millions to billions of dollars on pre-training, much less on fine-tuning, and nearly nothing per individual inference call (a single query→response exchange).

Inference-time scaling introduces a new paradigm: **significant compute can now be spent at inference time** to improve output quality — and critically, this inference compute **can be run offline**. An agent can be released to keep generating and refining candidate solutions to a problem without a human in the loop for each step.


# The Verification Bottleneck
Repeated sampling only works if there's a reliable way to pick the correct sample out of many candidates — i.e., **automated verification** is a prerequisite, not an afterthought.

Verification difficulty varies by domain:
- **Math (formal domains):** formal proof software can check each step of a proof strategy for correctness.
- **Coding:** unit tests serve as verifiers. Writing a unit test is often arguably simpler than writing the full program that solves the problem — making human-authored unit tests a practical verification source.

This sets up the natural next question: what happens in domains where good automated verifiers *don't* exist?


# AI as a Compiler: Another Verifiable Domain
One active research direction is using LLMs to generate lower-level compiled code — e.g., **CUDA** kernels from a higher-level source like PyTorch. CUDA is a lower-level, hardware-aware language used for writing GPU-optimized code.

## Why This Is Verifiable
Correctness can be checked directly: run the generated CUDA and the source PyTorch on the same inputs and compare outputs. If they match across inputs, the generated code is verified correct — no human judgment needed.

**KernelBench** (a CUDA-generation benchmark) shows the same coverage-vs-samples scaling pattern seen elsewhere, backed by this "perfect" automatic verifier.

## Generalization: Cross-Language Translation
The same idea extends to any code translation task (Python ↔ C++ ↔ Java, etc.) — equivalence between two programs' outputs is comparatively easy to measure, making these domains naturally well-suited to repeated sampling + verification.


# The Generation–Verification Gap
In domains **without** a reliable automated verifier, there's a large gap between what simple selection methods (like majority voting) achieve and the model's *true* coverage (what a perfect verifier could extract).

## Majority Voting vs. True Coverage
- **Majority voting:** among many generated answers, pick whichever answer appears most often.
- **True coverage (oracle/perfect verifier):** the upper bound — what's achievable if you could always correctly pick the right answer from the samples.

Plotting both against number of samples: majority voting **plateaus** after roughly 10–50 samples, while true coverage keeps climbing — a large and persistent gap. This gap widens further on harder datasets (e.g., MATH is harder than GSM8K, and shows a bigger gap).

Even learned selection methods — **outcome reward models** (LLM-based models trained to score response quality) or best-of-n approaches using them — still leave a substantial gap versus true coverage.

## Why Majority Voting Fails on Hard Problems
For the hardest problems, the *correct* answer may only appear once, twice, or three times out of 1,000–10,000 samples — far too rare to ever win a majority vote. Majority voting works reasonably well on simpler datasets (like GSM8K, where correct answers are more frequent among samples), but breaks down precisely where it matters most: the hardest problems, where correct samples are rare and hard to distinguish from confidently-wrong ones.


# Ways to Improve Verification
Some directions for closing the generation–verification gap:

- **Verifier quality is the bottleneck** — sampling alone helps little without a good verifier; the quality of the verifier directly bounds achievable accuracy.
- **Iterative revision** — having the model revise/critique its own prior solution across rounds (a preview of the sequential revision approach covered next).
- **Hybrid retrieval + ranking** — combining a knowledge-graph/RAG-style lookup with a ranking method, falling back to fuller documentation lookup only when confidence is low, as an efficiency/accuracy trade-off.
- **"Search before sampling"** — letting the model first explore/gather insights about a problem (a self-study phase) before generating final candidate solutions, potentially improving pass@k trends beyond naive parallel sampling. (This connects to later topics: self-study, search, and tool use as ways to extend test-time scaling beyond repeated sampling.)
- **Asymmetric verification** — in some domains, confirming an answer is *right* may be hard/expensive, but confirming an answer is *wrong* may be easier — suggesting a strategy of filtering out clearly-bad answers rather than requiring full verification. This idea generalizes to domains with imperfect verifiers, e.g., verifying against a simulation (physics, molecular dynamics) rather than a ground-truth checker.
- **Ensembling verifiers** — using multiple verifiers (e.g., 10–20) and combining their judgments (majority vote across verifiers), rather than relying on one. This is a real research direction (a related weakly-supervised verifier-ensembling method, "Weaver," is covered separately) — though compute cost scales with the number of verifiers used.

**Note:** the full 10,000-samples-per-problem dataset referenced here is publicly available on Hugging Face — closing the generation–verification gap is a strong potential research project topic.

## A Caution on Verifier Quality
Are "coverage" numbers trustworthy — could a solution pass unit tests without actually being correct? Manual checks on math problems found agreement in the high 90s (roughly 97–98%) — but this is not guaranteed in general. If unit tests don't have true/complete coverage of the intended behavior, verifier quality itself becomes a failure mode.


# Beyond Parallel Sampling: Sequential Revision
The Large Language Monkeys approach uses **parallel sampling** — many independent attempts, then select the best. But there's a second axis for spending test-time compute: **sequential revision**.

## Parallel vs. Sequential Test-Time Compute
| Approach | How it works |
|---|---|
| **Parallel sampling** | Generate many independent answers to the same input, then select the best via a verifier |
| **Sequential revision** | The model produces an initial attempt, then repeatedly revises/improves it — examining it from different angles — until confident, then outputs a final answer |

In the version discussed here, this revision behavior is driven by explicit prompting (asking the model to revise). Reasoning models, by contrast, are **trained** to exhibit this behavior internally — they naturally revise their own answers mid-generation without being explicitly prompted to do so.


# Two Ways to Score Candidate Answers: Outcome vs. Process Reward Models
Besides *how* answers are generated (parallel vs. sequential), there's also a choice in *how they're scored/selected* — via learned reward models rather than hard verifiers like unit tests.

- **Outcome Reward Models (ORMs):** trained to look at the **final answer** only and output a score/judgment of correctness. Useful when no hard verifier (like a unit test) exists, but accuracy can be limited — especially outside the model's training distribution/domain.
- **Process Reward Models (PRMs):** trained to score **each step** of the generated solution individually, rather than judging only the final output (e.g., scoring each of the 5 steps in a math derivation separately, not just the final number).

The key distinction: **outcome-based = score the final answer; process-based = score the reasoning trail itself.** Both are more general-purpose alternatives to exact verifiers, useful precisely where hard verification (unit tests, formal proofs) isn't available.


# Best-of-N Sampling with a Reward Model
The simplest way to bring a reward model into test-time scaling: generate N parallel samples, score each with an **outcome reward model (ORM)**, and take the highest-scoring one as the final answer.


# PRM-Guided Beam Search
Process reward models (PRMs) enable a more structured search than plain best-of-N: a **beam search over partial solutions**, guided step by step.

## How It Works
1. At each step of solving a problem, generate a fixed budget of samples (e.g., 4).
2. Score each partial continuation with the **PRM**.
3. Keep only the top-scoring subset (e.g., top 2) — a threshold/beam width choice.
4. From each surviving branch, sample again and repeat, expanding the tree only from the most promising continuations.

This is a **beam search** where the PRM decides which branches are worth continuing to expand, rather than exploring all branches equally.

## More About PRMs
- The PRM outputs a score (e.g., between 0 and 1) for a given step, conditioned on the question and/or the step itself.
- PRMs are typically **fine-tuned language models** themselves.
- They tend to perform best **in-domain** — training a PRM on a subset of a target benchmark improves test-time scaling on that specific benchmark more than it generalizes to new tasks — though as LM-based models, they do show some cross-task generalization.
- Scoring happens **per step**, not per token — a "step" might be defined as one intellectually meaningful chunk of the solution, or as simple as one sentence. Human annotators typically label step-level quality (good/bad) to train these models.


# Combining Outcome and Sequential/Parallel Approaches
Intuitively, combining outcome-based reward models with parallel *and* sequential revision strategies together outperforms any single approach alone — though this remains an active area with many open questions.

## One Simple Combination Strategy
Instead of doing either (a) one sample plus sequential revisions, or (b) many independent parallel samples, do **both**: generate multiple parallel samples, apply sequential revision to *each* one (augmenting each parallel branch with its own revision chain), then use a PRM or ORM to select the best final answer across all branches. A PRM can also guide the search *within* each sequential revision chain.

## Practical Note: Off-the-Shelf PRMs
Pretrained PRMs are available and can be applied directly to guide search on a new task, without training your own — though task-specific PRMs (trained on your own data) tend to perform better in-domain.


# Case Study: Difficulty-Dependent Scaling Behavior (MATH dataset)
A study using the MATH dataset (~12k train / 500 test questions) and the PaLM model investigated how optimal test-time scaling strategy depends on problem difficulty.

## Measuring Difficulty
Problems were binned into **5 difficulty levels**, defined by pass@1 performance — i.e., how many generated samples (out of a fixed budget) were correct for that problem. Bin 5 = hardest (lowest accuracy), Bin 1 = easiest.

## Key Observations
- **Majority voting**, **ORM-based selection**, and **PRM-based selection** were compared as generation budget increased — with a "mixed" approach (combining strategies) generally performing best, though how to optimally mix and match parallel vs. sequential compute remains an open research question. The underlying goal: for any target accuracy, minimize the generation budget needed to reach it.
- **Sequential-to-parallel ratio by difficulty:** for **easier problems**, allocating more compute to *sequential* revision (vs. parallel sampling) achieved the highest accuracy. For **harder problems**, the optimal ratio was much less clear-cut — even the trend between adjacent difficulty bins wasn't consistent.

## Test-Time Compute vs. Pre-Training Compute
Comparing the ratio of inference tokens to pre-training tokens against accuracy gains, split by difficulty:
- For **easy and medium** problems, additional test-time compute was **more favorable** than scaling up pre-training (i.e., more test-time compute beat spending that compute on a bigger pre-trained model).
- For the **hardest** problems, larger pre-trained models still won — even against a large test-time compute budget.

**Practical takeaway:** smaller/open-source models become increasingly competitive as more test-time compute is applied — but for the hardest problems, frontier models (more pre-training, larger scale) still have an edge, even under a large (if not literally infinite) test-time budget.

## Is Pre-Training Still Necessary?
If test-time scaling can close so much of the gap, is further pre-training investment still worthwhile? Yes, for two reasons — (1) for the hardest problems, pre-training still wins even against heavy test-time compute, and (2) **not everyone can afford to pre-train a large model**, so test-time scaling remains valuable precisely because it's a way to make existing, more accessible models more capable without the cost of full pre-training runs.

(Note: fine-tuning a smaller model to specialize on a narrow set of hard questions is possible, but the discussion here is about general-purpose training recipes on broad data, not narrow specialization.)


# Search Strategy and Compute Reuse
Mixing sequential revision with tree/beam search generally performs better than either approach alone, and can reduce redundant compute by reusing/pruning shared work across the tree — this connects directly to the PRM-guided beam search described earlier (cutting less-promising branches at each level), extended to also verify and prune *within* a chain-of-thought at each intermediate step, not just at the final answer.

**Framing for what's next:** think of generated tokens as a "knob" that trades compute for answer quality — the open design question is how to allocate that knob (sequential vs. parallel, which reward model, what beam width) and how to actually elicit strong generations from a model given a fixed budget.


# Archon: Inference-Time Architecture Search
**Paper:** [arxiv.org/abs/2409.15254](https://arxiv.org/abs/2409.15254)

Archon treats inference-time scaling as an **architecture design problem**: how do you mix and match different models and techniques to get the best possible accuracy for a given amount of compute (cost)?

## Framework Inputs and Output
- **Inputs:** a set of target benchmarks, an inference-call budget, a set of available LLMs (mixing multiple models is allowed), and a library of inference-time techniques.
- **Optimizer ("itest" — inference-time architecture search):** determines how to combine these pieces (which techniques, in what order, using which models) to maximize quality under the given budget.
- **Output:** a concrete inference architecture — a specific arrangement of techniques and models.


# Archon's Library of Inference-Time Techniques
All of these are **prompting-based** — no additional model training is involved.

- **Generation:** plain sampling from a model. Generating $n$ responses is the repeated-sampling building block.
- **Fusion:** given $K$ generated responses to a question, ask an LLM to synthesize **one** output response, showing it all $K$ candidates at once — letting it produce an answer informed by every attempted solution. Despite being simple, this turned out to be surprisingly effective.
- **Critic:** ask a model to describe the strengths/weaknesses of a given response.
- **Ranker:** ask a model to rank a set of generated responses by quality, along with its reasoning for the ranking (a prompted judgment, not a trained verifier).
- **Unit test generation & evaluation:** covered below.


# How Well Does Fusion Actually Work?
Measured as win rate on a reasoning/QA benchmark, varying the number of repeated samples (1→10):

| Method | Approach | Relative performance |
|---|---|---|
| Random selection | Pick one response at random | Worst |
| Model-based ranking | Rank responses, take the top-ranked one | Better than random |
| Oracle selection | A perfect verifier picks the best response | Better than ranking |
| **Fusion** | Show all $K$ responses to the model, synthesize one final answer | **Beats oracle selection** |
| Filter + fusion | Select the top 5 responses first, *then* fuse them | Best overall |

The standout result: fusion — just asking the model to synthesize a single answer from several candidates — can outperform even an oracle verifier that always picks the objectively best single response. Filtering down to a smaller, higher-quality set before fusing improves this further.

## Does This Hold Across Multiple Models?
Repeating the experiment with responses drawn from an ensemble of different models (1, 2, ... up to 10 different models, each producing one response) rather than one model producing all responses: random selection gets worse (since the single best model used on the left was one of the stronger ones in the ensemble), but the same overall ranking/trend among methods holds. Ensembles were built by starting with the best available model and adding progressively weaker ones.


# Unit Test Generation and Evaluation
Two more inference-time techniques, particularly relevant to coding and math/reasoning tasks:

- **Unit test generation:** ask the model to generate its own unit tests for a problem (e.g., for "check for balance of round brackets in a string" — a good generated test might check that a string with an odd number of brackets returns "no," or that a closing bracket must match the most recently unmatched opening bracket).
- **Unit test evaluation:** rather than actually *running* generated code against a unit test, ask the model to **evaluate** whether a generated answer satisfies a given unit test — using the model itself as the checker instead of an execution engine.


# What Architectures Did Archon Actually Find?
Archon-discovered architectures are structured as **layers** of these techniques stacked together — e.g., a first layer of generation (from an ensemble of different models), followed by a critic layer, a ranker, then one or more fuser layers, potentially repeating critic→ranker→fuser again before a final output.

## Search Space Reduction
Running the full search is expensive (many inference calls), so the search space was constrained using patterns found to work well in offline testing:
- Only **one inference-time technique per layer**.
- The **first layer is always a generator**.
- A **critic always precedes** a ranker or fuser (found empirically to work better).
- A **unit test generator must be followed by an evaluator**.

A **Bayesian optimizer** was used to search this constrained space against a held-out training dataset, optimizing accuracy for a given inference-call budget — more sample-efficient than greedy search or random selection, and able to target different objectives (available models, inference budget, etc.).

## Does Stacking More Layers Actually Help?
Yes — deeper architectures (more layers of critique/fusion, plus a model ensemble) significantly outperformed simpler setups (a single best model run once, or run 8 times with a single fusion layer) across many tasks. One way to think about it: much like adding layers in a deep learning architecture improves a pre-trained model, carefully adding inference-time layers improves accuracy at inference time.


# Archon's Results
A key framing detail: Archon still produces **one final answer** at the end — so it's optimizing **pass@1**, not coverage/pass@k.

## Headline Result
Using **only open-source models**, Archon-designed architectures matched or exceeded the frontier closed-source models of the time on pass@1, across many tasks — on average outperforming GPT-4o and Claude 3.5 Sonnet by **14.1 percentage points** when allowed to use any available model, and by **10.3 percentage points** when restricted to open-source models only. Benchmarks tested included MT-Bench, Arena-Hard-Auto, AlpacaEval 2.0, MixEval, MixEval Hard, MATH, and CodeContests.

## Task-Specific vs. General-Purpose Architectures
Archon can be optimized either:
- **Task-specific:** Bayesian optimization run and evaluated on a single target task.
- **General-purpose:** optimized to perform well across many tasks simultaneously.

Notably, the **general-purpose** architecture still outperformed frontier models across tasks — showing that a well-designed inference-time architecture can generalize well beyond the narrow set of tasks it was tuned on.
